# TripGraph — Notebook 2: Yelp Data → Neo4j Knowledge Graph

**Purpose:** Load the cleaned Yelp data produced by Notebook 1 into a Neo4j AuraDB graph database. This notebook is the data engineering bridge between the PySpark analytics (Part A) and the graph-based recommendation engine (Part B).

**Prerequisites:**
- Notebook 1 must have been run first — outputs written to `data/processed/`
- A free Neo4j AuraDB instance (create one at https://neo4j.com/cloud/aura-free)

---

## Graph Schema

```
(:User)-[:REVIEWED {stars, sentiment}]->(:Business)
(:User)-[:FRIENDS_WITH]->(:User)
(:Business)-[:IN_CATEGORY]->(:Category)
(:Business)-[:LOCATED_IN]->(:City)
(:City)-[:IN_STATE]->(:State)
```

## AuraDB Free Tier Limits

| Resource | Limit |
|---|---|
| Nodes | 200,000 |
| Relationships | 400,000 |
| Storage | 200 MB |

This notebook samples data to stay within these limits while preserving recommendation quality.

---
## Section 1 — Setup & Credentials

In [ ]:
# Install dependencies
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.0 neo4j

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

from google.colab import drive
drive.mount('/content/drive')

print('Setup complete.')

In [ ]:
# Neo4j AuraDB credentials
# Get these from your AuraDB instance dashboard after creating a free database.
# Store them securely — do NOT commit credentials to git.

from google.colab import userdata

# Option A: Colab Secrets (recommended) — add NEO4J_URI and NEO4J_PASSWORD
#           in the Colab sidebar under the key icon before running.
try:
    NEO4J_URI      = userdata.get('NEO4J_URI')
    NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
    NEO4J_USER     = 'neo4j'
    print('Credentials loaded from Colab Secrets.')
except Exception:
    # Option B: hardcode temporarily for local testing only
    NEO4J_URI      = 'neo4j+s://XXXXXXXX.databases.neo4j.io'  # replace
    NEO4J_PASSWORD = 'your-password-here'                      # replace
    NEO4J_USER     = 'neo4j'
    print('WARNING: Using hardcoded credentials — do not commit this file.')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, FloatType, IntegerType
from pyspark.sql.window import Window

import pandas as pd
import numpy as np
from neo4j import GraphDatabase
import time

print('Libraries imported.')

In [ ]:
spark = (
    SparkSession.builder
    .appName('TripGraph-Neo4jPipeline')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '20')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} ready.')

In [ ]:
# Paths — must match what Notebook 1 wrote
DRIVE_ROOT = '/content/drive/MyDrive/TripGraph'
OUT_DIR    = f'{DRIVE_ROOT}/data/processed'
RAW_DIR    = f'{DRIVE_ROOT}/data/yelp'

# AuraDB free tier limits
MAX_BUSINESS_NODES     = 15_000
MAX_USER_NODES         = 30_000
MAX_REVIEW_EDGES       = 120_000
MAX_FRIENDSHIP_EDGES   = 50_000

# Batch size for UNWIND queries (keep low to avoid timeout)
BATCH_SIZE = 500

TARGET_CITIES = [
    'Philadelphia', 'Nashville', 'Tampa', 'Indianapolis',
    'Tucson', 'Reno', 'New Orleans', 'Santa Barbara'
]

print(f'Processed data dir: {OUT_DIR}')

---
## Section 2 — Load Processed Data from Notebook 1

In [ ]:
# Load the four Parquet outputs written by Notebook 1
biz_df     = spark.read.parquet(f'{OUT_DIR}/businesses_clean')
biz_cat_df = spark.read.parquet(f'{OUT_DIR}/biz_categories')
review_df  = spark.read.parquet(f'{OUT_DIR}/reviews_clean')
user_df    = spark.read.parquet(f'{OUT_DIR}/users_clean')

print(f'Businesses : {biz_df.count():>8,}')
print(f'Biz-cats   : {biz_cat_df.count():>8,}')
print(f'Reviews    : {review_df.count():>8,}')
print(f'Users      : {user_df.count():>8,}')

In [ ]:
biz_df.printSchema()

---
## Section 3 — Prepare Neo4j-Ready DataFrames

We downsample each table to fit within AuraDB free tier limits while preserving the highest-quality nodes. The sampling strategy prioritises:
- Tourism-relevant businesses in target cities, ranked by quality score
- Users who reviewed those businesses (keeps the graph connected)
- Reviews that link sampled users to sampled businesses

### 3.1 Business Nodes

In [ ]:
# Keep tourism-relevant businesses in target cities, ranked by quality score.
# Fallback: if fewer than MAX_BUSINESS_NODES are tourism-relevant, include
# top non-tourism businesses to fill the quota.

biz_tourism = (
    biz_df
    .filter(F.col('is_tourism_relevant') == True)
    .filter(F.col('city').isin(TARGET_CITIES))
    .orderBy(F.desc('quality_score'))
    .limit(MAX_BUSINESS_NODES)
)

tourism_count = biz_tourism.count()
print(f'Tourism businesses selected: {tourism_count:,} / {MAX_BUSINESS_NODES:,} limit')

# Select final columns for the Business node
biz_nodes = (
    biz_tourism
    .select(
        F.col('business_id'),
        F.col('name'),
        F.col('city'),
        F.col('state'),
        F.col('latitude').alias('lat'),
        F.col('longitude').alias('lon'),
        F.col('stars'),
        F.col('review_count'),
        F.col('price_label'),
        F.col('quality_score'),
        F.col('avg_sentiment'),
        F.col('categories')
    )
    .fillna({'avg_sentiment': 0.0, 'quality_score': 0.0})
)

print(f'Business node columns: {biz_nodes.columns}')
biz_nodes.show(5, truncate=45)

### 3.2 Category & City Nodes

In [ ]:
# Extract unique categories from selected businesses
selected_biz_ids = biz_nodes.select('business_id')

category_nodes = (
    biz_cat_df
    .join(selected_biz_ids, on='business_id', how='inner')
    .select('category')
    .distinct()
    .orderBy('category')
)

city_nodes = (
    biz_nodes
    .select('city', 'state')
    .distinct()
    .orderBy('city')
)

print(f'Category nodes: {category_nodes.count():,}')
print(f'City nodes    : {city_nodes.count():,}')
category_nodes.show(10)
city_nodes.show()

### 3.3 User Nodes

In [ ]:
# Find users who reviewed our selected businesses
relevant_user_ids = (
    review_df
    .join(selected_biz_ids, on='business_id', how='inner')
    .select('user_id')
    .distinct()
)

# Join with user profile data; keep most active reviewers if over limit
user_nodes = (
    user_df
    .join(relevant_user_ids, on='user_id', how='inner')
    .orderBy(F.desc('review_count'))
    .limit(MAX_USER_NODES)
    .select(
        F.col('user_id'),
        F.col('review_count'),
        F.col('fans'),
        F.col('average_stars'),
        F.col('is_elite')
    )
    .fillna({'fans': 0, 'average_stars': 3.5})
)

print(f'User nodes selected: {user_nodes.count():,} / {MAX_USER_NODES:,} limit')
user_nodes.show(5)

### 3.4 Review Relationships

In [ ]:
# Reviews that link a selected user to a selected business
selected_user_ids = user_nodes.select('user_id')

review_edges = (
    review_df
    .join(selected_biz_ids, on='business_id', how='inner')
    .join(selected_user_ids, on='user_id', how='inner')
    .select(
        F.col('user_id'),
        F.col('business_id'),
        F.col('stars').alias('review_stars'),
        F.col('year')
    )
    # If a user reviewed the same business multiple times, keep the latest
    .withColumn('row_num',
        F.row_number().over(
            Window.partitionBy('user_id','business_id').orderBy(F.desc('year'))
        )
    )
    .filter(F.col('row_num') == 1)
    .drop('row_num')
    .orderBy(F.desc('year'))
    .limit(MAX_REVIEW_EDGES)
)

print(f'Review edges selected: {review_edges.count():,} / {MAX_REVIEW_EDGES:,} limit')
review_edges.show(5)

### 3.5 Business-Category & Business-City Relationships

In [ ]:
# Business → Category edges (one per category per business)
biz_category_edges = (
    biz_cat_df
    .join(selected_biz_ids, on='business_id', how='inner')
    .select('business_id', 'category')
    .distinct()
)

# Business → City edges (one per business)
biz_city_edges = biz_nodes.select('business_id', 'city').distinct()

print(f'Business-Category edges: {biz_category_edges.count():,}')
print(f'Business-City edges    : {biz_city_edges.count():,}')

biz_category_edges.show(5)
biz_city_edges.show(5)

### 3.6 Friendship Relationships

In [ ]:
# Load raw user data to extract the friends list
user_raw = spark.read.json(f'{RAW_DIR}/yelp_academic_dataset_user.json')

# Explode friends string into individual user_id pairs
friendship_edges = (
    user_raw
    .select('user_id', F.split(F.col('friends'), ', ').alias('friend_list'))
    .withColumn('friend_id', F.explode(F.col('friend_list')))
    .filter(F.col('friend_id') != 'None')
    .filter(F.col('friend_id') != '')
    .select('user_id', 'friend_id')
    # Keep only edges where BOTH users are in our selected set
    .join(selected_user_ids, on='user_id', how='inner')
    .join(selected_user_ids.toDF('friend_id'), on='friend_id', how='inner')
    .limit(MAX_FRIENDSHIP_EDGES)
)

print(f'Friendship edges selected: {friendship_edges.count():,} / {MAX_FRIENDSHIP_EDGES:,} limit')
friendship_edges.show(5)

### 3.7 Node & Edge Count Summary

In [ ]:
n_biz      = biz_nodes.count()
n_cat      = category_nodes.count()
n_city     = city_nodes.count()
n_user     = user_nodes.count()
n_total_nodes = n_biz + n_cat + n_city + n_user

n_reviewed     = review_edges.count()
n_biz_cat      = biz_category_edges.count()
n_biz_city     = biz_city_edges.count()
n_friends      = friendship_edges.count()
n_total_edges  = n_reviewed + n_biz_cat + n_biz_city + n_friends

print('Graph size summary:')
print(f'  Nodes      : {n_total_nodes:>8,}  (limit 200,000)')
print(f'    Business : {n_biz:>8,}')
print(f'    Category : {n_cat:>8,}')
print(f'    City     : {n_city:>8,}')
print(f'    User     : {n_user:>8,}')
print()
print(f'  Edges      : {n_total_edges:>8,}  (limit 400,000)')
print(f'    REVIEWED     : {n_reviewed:>8,}')
print(f'    IN_CATEGORY  : {n_biz_cat:>8,}')
print(f'    LOCATED_IN   : {n_biz_city:>8,}')
print(f'    FRIENDS_WITH : {n_friends:>8,}')

if n_total_nodes > 200_000:
    print('\nWARNING: Node count exceeds AuraDB free tier limit. Reduce MAX constants above.')
if n_total_edges > 400_000:
    print('\nWARNING: Edge count exceeds AuraDB free tier limit. Reduce MAX constants above.')
else:
    print('\nWithin AuraDB free tier limits.')

### 3.8 Export to CSV (for local reference / backup)

In [ ]:
# Convert to Pandas — used both for CSV export and for Neo4j loading
biz_pd         = biz_nodes.toPandas()
cat_pd         = category_nodes.toPandas()
city_pd        = city_nodes.toPandas()
user_pd        = user_nodes.toPandas()
review_pd      = review_edges.toPandas()
biz_cat_pd     = biz_category_edges.toPandas()
biz_city_pd    = biz_city_edges.toPandas()
friendship_pd  = friendship_edges.toPandas()

NEO4J_DIR = f'{DRIVE_ROOT}/data/neo4j'
os.makedirs(NEO4J_DIR, exist_ok=True)

biz_pd.to_csv(        f'{NEO4J_DIR}/businesses.csv',    index=False)
cat_pd.to_csv(        f'{NEO4J_DIR}/categories.csv',    index=False)
city_pd.to_csv(       f'{NEO4J_DIR}/cities.csv',        index=False)
user_pd.to_csv(       f'{NEO4J_DIR}/users.csv',         index=False)
review_pd.to_csv(     f'{NEO4J_DIR}/reviews.csv',       index=False)
biz_cat_pd.to_csv(    f'{NEO4J_DIR}/biz_categories.csv',index=False)
biz_city_pd.to_csv(   f'{NEO4J_DIR}/biz_cities.csv',    index=False)
friendship_pd.to_csv( f'{NEO4J_DIR}/friendships.csv',   index=False)

print(f'CSVs written to: {NEO4J_DIR}')

---
## Section 4 — Connect to Neo4j AuraDB

In [ ]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Verify connection
with driver.session() as session:
    result = session.run('RETURN "Connected to Neo4j AuraDB" AS message, datetime() AS ts')
    row = result.single()
    print(row['message'])
    print(f'Server time: {row["ts"]}')

---
## Section 5 — Schema: Constraints & Indexes

Constraints enforce uniqueness and create the underlying indexes that make lookups fast. They must be created before loading data.

In [ ]:
CONSTRAINTS = [
    'CREATE CONSTRAINT business_id IF NOT EXISTS FOR (b:Business) REQUIRE b.business_id IS UNIQUE',
    'CREATE CONSTRAINT user_id     IF NOT EXISTS FOR (u:User)     REQUIRE u.user_id IS UNIQUE',
    'CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE',
    'CREATE CONSTRAINT city_name   IF NOT EXISTS FOR (ci:City)    REQUIRE ci.name IS UNIQUE',
]

INDEXES = [
    'CREATE INDEX business_city   IF NOT EXISTS FOR (b:Business) ON (b.city)',
    'CREATE INDEX business_stars  IF NOT EXISTS FOR (b:Business) ON (b.stars)',
    'CREATE INDEX business_quality IF NOT EXISTS FOR (b:Business) ON (b.quality_score)',
    'CREATE INDEX business_price  IF NOT EXISTS FOR (b:Business) ON (b.price_label)',
    'CREATE INDEX user_elite      IF NOT EXISTS FOR (u:User)     ON (u.is_elite)',
]

with driver.session() as session:
    for stmt in CONSTRAINTS:
        session.run(stmt)
        print(f'  OK: {stmt[:70]}...')
    for stmt in INDEXES:
        session.run(stmt)
        print(f'  OK: {stmt[:70]}...')

print('\nAll constraints and indexes created.')

---
## Section 6 — Load Nodes into Neo4j

We load in batches using `UNWIND` — each batch sends 500 rows in a single transaction, which is far more efficient than one query per row.

In [ ]:
def load_in_batches(session, query, records, batch_size=BATCH_SIZE, label='records'):
    """Send records to Neo4j in batches via UNWIND."""
    total   = len(records)
    batches = range(0, total, batch_size)
    for i, start in enumerate(batches):
        chunk = records[start : start + batch_size]
        session.run(query, rows=chunk)
        if (i + 1) % 10 == 0 or start + batch_size >= total:
            print(f'  {min(start + batch_size, total):>6,} / {total:,} {label} loaded')
    return total

print('Batch loader ready.')

### 6.1 Business Nodes

In [ ]:
MERGE_BUSINESS = """
UNWIND $rows AS row
MERGE (b:Business {business_id: row.business_id})
SET
  b.name          = row.name,
  b.city          = row.city,
  b.state         = row.state,
  b.lat           = toFloat(row.lat),
  b.lon           = toFloat(row.lon),
  b.stars         = toFloat(row.stars),
  b.review_count  = toInteger(row.review_count),
  b.price_label   = row.price_label,
  b.quality_score = toFloat(row.quality_score),
  b.avg_sentiment = toFloat(row.avg_sentiment),
  b.categories    = row.categories
"""

records = biz_pd.where(biz_pd.notna()).to_dict('records')

t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_BUSINESS, records, label='businesses')
print(f'Loaded {n:,} Business nodes in {time.time()-t0:.1f}s')

### 6.2 Category Nodes

In [ ]:
MERGE_CATEGORY = """
UNWIND $rows AS row
MERGE (c:Category {name: row.category})
"""

records = cat_pd.to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_CATEGORY, records, label='categories')
print(f'Loaded {n:,} Category nodes in {time.time()-t0:.1f}s')

### 6.3 City Nodes

In [ ]:
MERGE_CITY = """
UNWIND $rows AS row
MERGE (ci:City {name: row.city})
SET ci.state = row.state
"""

records = city_pd.to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_CITY, records, label='cities')
print(f'Loaded {n:,} City nodes in {time.time()-t0:.1f}s')

### 6.4 User Nodes

In [ ]:
MERGE_USER = """
UNWIND $rows AS row
MERGE (u:User {user_id: row.user_id})
SET
  u.review_count  = toInteger(row.review_count),
  u.fans          = toInteger(row.fans),
  u.average_stars = toFloat(row.average_stars),
  u.is_elite      = row.is_elite
"""

records = user_pd.where(user_pd.notna()).to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_USER, records, label='users')
print(f'Loaded {n:,} User nodes in {time.time()-t0:.1f}s')

---
## Section 7 — Load Relationships into Neo4j

### 7.1 REVIEWED Relationships (User → Business)

In [ ]:
MERGE_REVIEWED = """
UNWIND $rows AS row
MATCH (u:User     {user_id:     row.user_id})
MATCH (b:Business {business_id: row.business_id})
MERGE (u)-[r:REVIEWED]->(b)
SET
  r.stars = toFloat(row.review_stars),
  r.year  = toInteger(row.year)
"""

records = review_pd.where(review_pd.notna()).to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_REVIEWED, records, label='REVIEWED edges')
print(f'Loaded {n:,} REVIEWED relationships in {time.time()-t0:.1f}s')

### 7.2 IN_CATEGORY Relationships (Business → Category)

In [ ]:
MERGE_IN_CATEGORY = """
UNWIND $rows AS row
MATCH (b:Business {business_id: row.business_id})
MATCH (c:Category {name:        row.category})
MERGE (b)-[:IN_CATEGORY]->(c)
"""

records = biz_cat_pd.to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_IN_CATEGORY, records, label='IN_CATEGORY edges')
print(f'Loaded {n:,} IN_CATEGORY relationships in {time.time()-t0:.1f}s')

### 7.3 LOCATED_IN Relationships (Business → City)

In [ ]:
MERGE_LOCATED_IN = """
UNWIND $rows AS row
MATCH (b:Business {business_id: row.business_id})
MATCH (ci:City    {name:        row.city})
MERGE (b)-[:LOCATED_IN]->(ci)
"""

records = biz_city_pd.to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_LOCATED_IN, records, label='LOCATED_IN edges')
print(f'Loaded {n:,} LOCATED_IN relationships in {time.time()-t0:.1f}s')

### 7.4 FRIENDS_WITH Relationships (User → User)

In [ ]:
MERGE_FRIENDS = """
UNWIND $rows AS row
MATCH (u1:User {user_id: row.user_id})
MATCH (u2:User {user_id: row.friend_id})
MERGE (u1)-[:FRIENDS_WITH]->(u2)
"""

records = friendship_pd.to_dict('records')
t0 = time.time()
with driver.session() as session:
    n = load_in_batches(session, MERGE_FRIENDS, records, label='FRIENDS_WITH edges')
print(f'Loaded {n:,} FRIENDS_WITH relationships in {time.time()-t0:.1f}s')

---
## Section 8 — Verification Queries

In [ ]:
def run_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, **(params or {}))
        return [dict(r) for r in result]

### 8.1 Node & Relationship Counts

In [ ]:
count_queries = [
    ('Business nodes',   'MATCH (n:Business) RETURN count(n) AS count'),
    ('Category nodes',   'MATCH (n:Category) RETURN count(n) AS count'),
    ('City nodes',       'MATCH (n:City)     RETURN count(n) AS count'),
    ('User nodes',       'MATCH (n:User)     RETURN count(n) AS count'),
    ('REVIEWED rels',    'MATCH ()-[r:REVIEWED]->()     RETURN count(r) AS count'),
    ('IN_CATEGORY rels', 'MATCH ()-[r:IN_CATEGORY]->()  RETURN count(r) AS count'),
    ('LOCATED_IN rels',  'MATCH ()-[r:LOCATED_IN]->()   RETURN count(r) AS count'),
    ('FRIENDS_WITH rels','MATCH ()-[r:FRIENDS_WITH]->()  RETURN count(r) AS count'),
]

print(f'{"Item":<22} {"Count":>10}')
print('-' * 34)
for label, q in count_queries:
    c = run_query(q)[0]['count']
    print(f'{label:<22} {c:>10,}')

### 8.2 Sample: Top 10 Businesses in Philadelphia by Quality Score

In [ ]:
q = """
MATCH (b:Business)-[:LOCATED_IN]->(ci:City {name: 'Philadelphia'})
RETURN b.name AS name, b.stars AS stars, b.review_count AS reviews,
       b.quality_score AS quality, b.price_label AS price
ORDER BY b.quality_score DESC
LIMIT 10
"""
rows = run_query(q)
print('Top 10 Philadelphia businesses by quality score:')
print(pd.DataFrame(rows).to_string(index=False))

### 8.3 Sample: Business → Category Paths

In [ ]:
q = """
MATCH (b:Business)-[:IN_CATEGORY]->(c:Category)
WHERE b.city = 'Nashville'
RETURN b.name AS business, collect(c.name) AS categories
ORDER BY b.quality_score DESC
LIMIT 8
"""
rows = run_query(q)
print('Nashville businesses and their categories:')
for row in rows:
    cats = ', '.join(row['categories'][:5])
    print(f"  {row['business'][:40]:<40}  [{cats}]")

### 8.4 Sample: Users Who Reviewed Multiple Tourism Categories

In [ ]:
q = """
MATCH (u:User)-[:REVIEWED]->(b:Business)-[:IN_CATEGORY]->(c:Category)
RETURN u.user_id AS user_id,
       count(DISTINCT b)      AS businesses_reviewed,
       count(DISTINCT c.name) AS categories_visited,
       u.is_elite             AS is_elite
ORDER BY businesses_reviewed DESC
LIMIT 10
"""
rows = run_query(q)
print('Most active reviewers across tourism categories:')
print(pd.DataFrame(rows).to_string(index=False))

### 8.5 Sample: 3-Hop Path (User → Business → Category ← Business)

In [ ]:
# Find a business similar to a seed business via shared category
q = """
MATCH (seed:Business {city: 'Philadelphia'})
WITH seed ORDER BY seed.quality_score DESC LIMIT 1
MATCH (seed)-[:IN_CATEGORY]->(c:Category)<-[:IN_CATEGORY]-(similar:Business)
WHERE similar <> seed AND similar.city = seed.city
RETURN seed.name AS seed_business,
       c.name    AS shared_category,
       similar.name         AS similar_business,
       similar.quality_score AS quality
ORDER BY similar.quality_score DESC
LIMIT 5
"""
rows = run_query(q)
print('Businesses similar to top Philadelphia business via shared category:')
print(pd.DataFrame(rows).to_string(index=False))

### 8.6 Graph Statistics

In [ ]:
# Avg degree — how many reviews does a business have in the graph?
q_biz_degree = """
MATCH (b:Business)
OPTIONAL MATCH (b)<-[r:REVIEWED]-()
RETURN
  avg(count(r)) AS avg_review_degree,
  max(count(r)) AS max_review_degree,
  min(count(r)) AS min_review_degree
"""

q_city_dist = """
MATCH (b:Business)-[:LOCATED_IN]->(ci:City)
RETURN ci.name AS city, count(b) AS business_count
ORDER BY business_count DESC
"""

deg = run_query(q_biz_degree)[0]
print('Business review degree in graph:')
print(f'  Avg : {deg["avg_review_degree"]:.1f}')
print(f'  Max : {deg["max_review_degree"]}')
print(f'  Min : {deg["min_review_degree"]}')

print('\nBusiness count per city in graph:')
city_dist = run_query(q_city_dist)
print(pd.DataFrame(city_dist).to_string(index=False))

---
## Section 9 — Neo4j GDS: Install Graph Catalog Entry

Before running graph algorithms in Notebook 3, we project the relevant subgraph into the GDS in-memory catalog. This is done here to verify the projection works correctly.

In [ ]:
# Drop existing projection if it exists (safe to re-run)
try:
    with driver.session() as session:
        session.run("CALL gds.graph.drop('tripgraph', false) YIELD graphName")
    print('Dropped existing tripgraph projection.')
except Exception:
    print('No existing projection to drop.')

# Project Business + User nodes with REVIEWED relationship
# This is the subgraph used for Personalized PageRank and Node Similarity
PROJECT_QUERY = """
CALL gds.graph.project(
  'tripgraph',
  {
    Business: { properties: ['quality_score', 'stars', 'avg_sentiment'] },
    User:     { properties: ['review_count', 'is_elite'] }
  },
  {
    REVIEWED: {
      type: 'REVIEWED',
      orientation: 'UNDIRECTED',
      properties: ['stars']
    },
    IN_CATEGORY: {
      type: 'IN_CATEGORY',
      orientation: 'UNDIRECTED'
    }
  }
)
YIELD graphName, nodeCount, relationshipCount
"""

result = run_query(PROJECT_QUERY)
if result:
    r = result[0]
    print(f"Graph projection 'tripgraph' created:")
    print(f"  Nodes         : {r['nodeCount']:,}")
    print(f"  Relationships : {r['relationshipCount']:,}")
else:
    print('Projection returned no result — check GDS is enabled on your AuraDB instance.')

In [ ]:
# Quick smoke-test: run Degree Centrality to confirm GDS is working
DEGREE_TEST = """
CALL gds.degree.stream('tripgraph', {nodeLabels: ['Business']})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS business, score AS degree
ORDER BY degree DESC
LIMIT 5
"""

rows = run_query(DEGREE_TEST)
if rows:
    print('Top 5 businesses by degree centrality (GDS working):')
    print(pd.DataFrame(rows).to_string(index=False))
else:
    print('No results — GDS may not be enabled on this AuraDB tier.')

---
## Section 10 — Cleanup & Summary

In [ ]:
# Final graph state summary
summary_q = """
CALL apoc.meta.stats()
YIELD nodeCount, relCount, labels, relTypesCount
RETURN nodeCount, relCount, labels, relTypesCount
"""

# apoc.meta.stats is available on AuraDB; if not, fall back to manual counts
try:
    summary = run_query(summary_q)
    r = summary[0]
    print(f'Total nodes         : {r["nodeCount"]:,}')
    print(f'Total relationships : {r["relCount"]:,}')
    print(f'Node labels         : {list(r["labels"].keys())}')
    print(f'Relationship types  : {list(r["relTypesCount"].keys())}')
except Exception:
    # apoc not available — use manual counts already printed in Section 8
    print('apoc.meta.stats not available; see Section 8 counts above.')

print('\nGraph is ready for Notebook 3 (recommendation engine).')

In [ ]:
# Close connections
driver.close()
spark.stop()
print('Neo4j driver and Spark session closed. Notebook 2 complete.')